# Validation v2: frozen CalibratedTotalRisk

This notebook evaluates the Sprint 1 primary revised model without model selection or temporal refitting. It verifies the three canonical SHA256 identities before analysis. `CalibratedTotalRisk` uses the locked training-only coefficients `beta_Demo=1.0586578127615558` and `beta_Physio=0.3563514638575306`.

Historical submitted values remain separately labeled. Cox fits use explicit step-size/iteration/precision controls, and convergence warnings are captured locally rather than suppressed. Bootstrap seed: **20260821**.

Uno's IPCW C and cumulative/dynamic AUC use `scikit-survival` 0.28.0, with training data estimating the censoring distribution and temporal data used for evaluation. Fixed-horizon IPCW Brier scores use the temporal censoring distribution as an evaluation metric.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'scripts'))
import run_sprint2_validation as s2
print({'bootstrap_seed': s2.SEED, 'bootstrap_replicates': s2.N_BOOT, 'data_dir': str(s2.DATA_DIR)})

{'bootstrap_seed': 20260821, 'bootstrap_replicates': 2000, 'data_dir': '/home/ubuntu2204/longevity/data/physiorisk/data'}


The exported validation table contains discrimination, uncertainty, HR per actual test-cohort SD, HR confidence intervals, and age correlation. Paired resampling uses identical bootstrap participants for both scores in each contrast.

In [2]:
validation = s2.run_validation()
display(pd.read_csv(ROOT/'tables/validation_v2.csv'))
display(pd.read_csv(ROOT/'tables/cindex_bootstrap_contrasts.csv'))

,model,value_provenance,cohort,N,deaths,C_index,C_index_bootstrap_95CI_low,C_index_bootstrap_95CI_high,bootstrap_replicates,bootstrap_seed,actual_test_SD,HR_per_actual_1SD,HR_95CI_low,HR_95CI_high,age_correlation,cox_penalizer,cox_convergence_warning
0,DemoRisk,recomputed/revised,temporal_test,6739,416,0.826876,0.804018,0.846937,2000.0,20260821.0,1.308431,3.735674,3.346769,4.169771,0.969734,0.0,none
1,PhysioRisk_submitted,submitted/historical,temporal_test,6739,416,0.654110,NaN,NaN,NaN,NaN,NaN,2.771658,2.655681,2.892700,0.106233,0.0,historical fit did not converge; preserved only
2,PhysioRisk_convergence_corrected,recomputed/revised,temporal_test,6739,416,0.654110,0.624283,0.685027,2000.0,20260821.0,0.949861,1.387468,1.337516,1.439286,0.106233,0.0,none
3,TotalRisk_original,recomputed/revised,temporal_test,6739,416,0.839760,0.818611,0.860145,2000.0,20260821.0,1.689100,1.877466,1.815234,1.941831,0.810927,0.0,none
4,CalibratedTotalRisk,recomputed/revised,temporal_test,6739,416,0.843153,0.821796,0.862708,2000.0,20260821.0,1.457188,3.259002,3.035997,3.498387,0.946491,0.0,none
5,PhenoAge NCP,recomputed/revised,temporal_test,6739,416,0.841108,0.819117,0.861020,2000.0,20260821.0,1.666348,3.976116,3.636923,4.346944,0.961369,0.0,none
6,PhenoAge NCP acceleration,recomputed/revised,temporal_test,6739,416,0.637857,0.605876,0.667943,2000.0,20260821.0,0.458601,1.432147,1.359932,1.508198,0.033299,0.0,none


,contrast,model_A,model_B,value_provenance,cohort,point_C_index_difference_A_minus_B,bootstrap_95CI_low,bootstrap_95CI_high,proportion_bootstrap_difference_above_zero,bootstrap_replicates,bootstrap_seed
0,CalibratedTotalRisk vs DemoRisk,CalibratedTotalRisk,DemoRisk,recomputed/revised,temporal_test,0.016277,0.011092,0.021987,1.000,2000,20260821
1,CalibratedTotalRisk vs PhenoAge NCP,CalibratedTotalRisk,PhenoAge NCP,recomputed/revised,temporal_test,0.002045,-0.003258,0.007480,0.776,2000,20260821
2,PhysioRisk_convergence_corrected vs PhenoAge N...,PhysioRisk_convergence_corrected,PhenoAge NCP acceleration,recomputed/revised,temporal_test,0.016252,-0.000494,0.032851,0.970,2000,20260821


PH diagnostics are descriptive. A flagged result is recorded, not used to alter the model.

In [3]:
display(pd.read_csv(ROOT/'tables/ph_diagnostics.csv'))

,model,value_provenance,cohort,N,deaths,time_transform,test_statistic,p_value,material_violation_flag,flag_rule,cox_convergence_warning,scaled_residual_region_summary
0,DemoRisk,recomputed/revised,temporal_test,6739,416,rank,2.925210,0.087206,False,p<0.01 (prespecified descriptive flag; no mult...,none,early_0_36m: mean=-0.087; middle_37_72m: mean=...
1,PhysioRisk,recomputed/revised,temporal_test,6739,416,rank,2.466846,0.116271,False,p<0.01 (prespecified descriptive flag; no mult...,none,early_0_36m: mean=-0.035; middle_37_72m: mean=...
2,CalibratedTotalRisk,recomputed/revised,temporal_test,6739,416,rank,16.539455,0.000048,True,p<0.01 (prespecified descriptive flag; no mult...,none,early_0_36m: mean=-0.177; middle_37_72m: mean=...
3,HematologicStress,recomputed/revised,temporal_test,6739,416,rank,3.852417,0.049674,False,p<0.01 (prespecified descriptive flag; no mult...,none,NaN
4,RenalHepaticTurnover,recomputed/revised,temporal_test,6739,416,rank,4.303924,0.038025,False,p<0.01 (prespecified descriptive flag; no mult...,none,NaN


Absolute risk uses the training Cox baseline survival and frozen training coefficients. Temporal outcomes are used only for evaluation. The eight-year horizon is retained with an explicit less-precise-tail qualification.

In [4]:
display(pd.read_csv(ROOT/'tables/followup_summary.csv'))
display(pd.read_csv(ROOT/'tables/calibration_fixed_horizons.csv'))

,cohort,value_provenance,N,deaths_total,followup_definition,reverse_KM_median_months,empirical_median_months,empirical_Q1_months,empirical_Q3_months,maximum_months,horizon_months,number_at_risk,deaths_observed_by_horizon,support_assessment
0,temporal_test,recomputed/revised,6739,416,reverse_KM_median; empirical_IQR_and_max,74.0,71.0,53.0,90.0,111.0,60,4461,289,supported
1,temporal_test,recomputed/revised,6739,416,reverse_KM_median; empirical_IQR_and_max,74.0,71.0,53.0,90.0,111.0,96,1137,410,supported_with_less_precise_tail


,model,value_provenance,cohort,horizon_months,training_baseline_survival,mean_predicted_mortality_risk,KM_observed_mortality,calibration_in_large_observed_minus_predicted,observed_to_expected_ratio,calibration_slope,calibration_slope_95CI_low,calibration_slope_95CI_high,global_calibration_slope_method,horizon_specific_calibration_intercept,horizon_specific_calibration_slope,horizon_specific_calibration_method,IPCW_Brier_score,IPCW_weight_source,IPCW_usable_N,no_temporal_recalibration
0,CalibratedTotalRisk,recomputed/revised,temporal_test,60,0.983687,0.046101,0.046533,0.000432,1.009370,0.810754,0.762112,0.859396,temporal Cox coefficient on frozen LP; assumes PH,-0.101177,0.941022,IPCW-weighted logistic calibration on logit(fr...,0.037758,temporal-test censoring KM (evaluation only),4709,True
1,CalibratedTotalRisk,recomputed/revised,temporal_test,96,0.966595,0.085710,0.088256,0.002546,1.029704,0.810754,0.762112,0.859396,temporal Cox coefficient on frozen LP; assumes PH,0.066545,0.999029,IPCW-weighted logistic calibration on logit(fr...,0.060177,temporal-test censoring KM (evaluation only),1518,True


Calibration plot data are exported as temporal-test risk deciles; plotting can be styled later without rerunning the model.

In [5]:
display(pd.read_csv(ROOT/'tables/calibration_plot_data.csv'))

,model,value_provenance,cohort,horizon_months,risk_bin,N,deaths_total,mean_predicted_risk,KM_observed_risk,number_at_risk_at_horizon
0,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(1.0000000000000026e-05, 0.00308]",674,3,0.002443,0.004555,484
1,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.00308, 0.00425]",674,5,0.003650,0.007434,469
2,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.00425, 0.00581]",674,6,0.004963,0.005935,481
3,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.00581, 0.00818]",674,5,0.006829,0.008381,475
4,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.00818, 0.0122]",674,15,0.009961,0.013220,457
5,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.0122, 0.0202]",673,14,0.015821,0.016819,464
6,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.0202, 0.0352]",674,24,0.026726,0.026884,443
7,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.0352, 0.0631]",674,47,0.047372,0.051445,432
8,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.0631, 0.137]",674,78,0.092433,0.079863,407
9,CalibratedTotalRisk,recomputed/revised,temporal_test,60,"(0.137, 1.0]",674,219,0.250764,0.252688,349


Scaled Schoenfeld residuals and an extended-Cox score × log(time/60 months) model localize non-proportionality. These are diagnostic only. Horizon-specific IPCW logistic calibration slopes are kept distinct from the global PH-dependent Cox slope.